# Object Detection (YOLOv2 / PASCAL VOC)

<p align="right">
Run Time: ~10 minutes
</p>

This notebook walks through benchmarking a model on the **PASCAL VOC**
('car' / 'person') detection task, targeting Akida 1 hardware.

For details on the dataset and preparation of the model, see the neighbouring README.md
and detection_notebook_training.ipynb.

## Model

A pretrained Akida model is expected in this repo, at
`pretrained_models/yolo_akidanet_detection_qat.fbz`. As with all model files in the repo,
that is handled via `git-lfs` (for efficient large file storage). If you haven't set that
up yet, see the [Trained models](../../../README.md#trained-models) section of the
top-level README.

If you have run through the training scripts or notebook, and prefer to benchmark the
Akida model generated through that, simply modify the model directory and filename in the
following cell:

In [ ]:
import akida
import os
import numpy as np

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '1')

DATA_PATH = './data/voc'
MODELS_DIR = './pretrained_models/'
MODEL_FILENAME = 'yolo_akidanet_detection_qat.fbz'
akida_model_path = os.path.join(MODELS_DIR, MODEL_FILENAME)

SEED = 42

To load the model, we simply have to pass the path to the `.fbz` file to the `akida.Model()`
method:

In [ ]:
# Load the Akida model
akida_model = akida.Model(akida_model_path)

## Dataset

`get_voc_dataset` returns the raw (unprocessed) VOC validation split, which the
mAP evaluator needs directly; `get_anchors` returns the fixed anchor boxes used
to decode model predictions. The full preprocessing/augmentation code is in
[detection_data.py](detection_data.py) and `akida_models.detection`.

Images are resized from variable original sizes to **224 x 224 RGB** and delivered
as pixel values in the uint8 range (0-255).

In [ ]:
from detection_data import get_anchors, LABELS
from akida_models.detection.voc.data import get_voc_dataset

INPUT_SHAPE = (224, 224, 3)

anchors = get_anchors()
val_data_raw, labels, num_valid = get_voc_dataset(DATA_PATH, labels=LABELS, training=False)

## Evaluation of Akida Model

We now run mAP evaluation through the Akida model, to check that it is
comparable to that obtained from the quantized tf_keras model. If an Akida 1
hardware device is connected, it will be used for inference; if not, the
code will fall back to using the software backend: this delivers a
bit-accurate simulation of the results that will be obtained when running
the model on hardware. Let's run that check before going any further

### Check for a connected Akida hardware device

We can use the `akida.devices()` function to detect connected hardware devices.
That returns a list - if it's empty, there were no hardware devices. Otherwise,
typically we'd only have a single Akida device connected on a given machine,
and we can just select the first (and only) device returned.

In [ ]:
devices = akida.devices()
if len(devices)>0:
    # Hardware is available
    device = devices[0]
else:
    # Hardware is not available
    device = None

In the present case, we want to be a bit more careful and ensure that the
device is the right version for the model we want to test (here, Akida IP
version 1). We'll import a local function to do that - check out the details
if interested

In [ ]:
from brainchip_utils.hardware_utils import get_akida_device

# Look for a matching hardware device
device = get_akida_device(target_version = akida_model.ip_version)
if device is not None:
    akida_model.map(device, mode=akida.MapMode.Minimal)

### Run mAP Evaluation on Akida

`MapEvaluation` (from `akida_models.detection`) manually iterates the raw
validation data, resizing/decoding each image with the model's own input
shape and anchors, and runs inference through the Akida model directly
(whichever backend it is mapped to) - no manual batching or reshaping is
needed on our side.

mAP is computed for IoU thresholds from 0.5 to 0.95 (step 0.05) and averaged
across thresholds and classes.

In [ ]:
from akida_models.detection.map_evaluation import MapEvaluation

map_evaluator = MapEvaluation(akida_model, val_data_raw, num_valid, labels, anchors,
                              is_keras_model=False)
map_dict, average_precisions = map_evaluator.evaluate_map()
akida_map = sum(map_dict.values()) / len(map_dict)
print(f'mAP 50: {map_dict[0.5]:.4f}')
print(f'mAP 75: {map_dict[0.75]:.4f}')
for label, average_precision in average_precisions.items():
    print(f'{labels[label]}: {average_precision:.4f}')
print(f'Akida mAP: {akida_map:.4f}')

### Activation Sparsity

Akida hardware skips computation for zero-valued activations, so activation
sparsity directly reduces both energy consumption and inference latency.
Below we measure per-layer sparsity on a calibration batch drawn from the
validation set.

In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity
from detection_data import get_samples

NUM_SAMPLES = 1000
samples = get_samples(DATA_PATH, INPUT_SHAPE, num_samples=NUM_SAMPLES)
sparsity_dict = compute_sparsity(akida_model, samples=samples)
pretty_print_sparsity(sparsity_dict)

## Hardware Benchmarking

**These cells require a physical AKD1500 device to be connected.** If `device is
None` (reported in the evaluation section above), there is nothing further to run.

Akida is an event-driven architecture: computations scale with the number of
non-zero activations, not with tensor size. That means benchmark results are
*input-dependent* - random or synthetic data would give artificially fast or
slow timings. The `samples` array loaded above (real images from the validation
split) is therefore the correct input to use here.

### Simple Benchmark

The simplest way to time an Akida model is to call `forward` in a loop and
read back two clocks after each inference:

- **System clock** (`time.perf_counter_ns`) - wall time including Python and
  USB/PCIe transfer overhead.
- **On-chip clock** (`akida_model.metrics['inference_clk']`) - raw clock cycles
  counted by the AKD1500 itself. Dividing by the 400 MHz core frequency gives
  the pure compute time.

The two numbers should agree closely; a large divergence would indicate a
transfer or driver bottleneck.

In [ ]:
import time

CLOCK_FREQUENCY = 400e6  # 400 MHz for AKD1500

if device is not None:
    akida_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)

    # Run a priming frame, so that the model is loaded to the device
    # and the subsequent benchmarking reflects inference time only
    akida_model.forward(samples[:2])
    
    inf_clks = []
    inf_times = []
    for i in range(len(samples)):
        start_t = time.perf_counter_ns()
        akida_model.forward(samples[i:i+1])
        inf_times.append(time.perf_counter_ns() - start_t)
        inf_clks.append(akida_model.metrics['inference_clk'])

    mean_inf_clk = np.mean(inf_clks) / CLOCK_FREQUENCY * 1e3  # cycles → ms
    mean_inf_time = np.mean(inf_times) * 1e-6                  # ns → ms
    print(f'Mean inference time (system clock):      {mean_inf_time:.3f} ms')
    print(f'Mean on-chip time (chip clock cycles):   {mean_inf_clk:.3f} ms')

### Full Model Benchmark

The loop above is clear, but it misses two things: power consumption and a
comparison between mapping modes. `full_model_benchmark` from
[brainchip_utils/hardware_utils.py](../../../brainchip_utils/hardware_utils.py)
runs the same timed loop while also coordinating optional INA219 power
measurement in a separate process. It sweeps both `MapMode.Minimal` (fewest
NPs, lowest power) and `MapMode.AllNps` (all NPs, maximum parallelism) so the
trade-off is visible. The multiprocessing and power-meter wiring are
non-trivial and not of interest to most users - consult the source if needed.

In [ ]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats
from brainchip_utils.plot_utils import plot_full_model_results

if device is not None:
    map_modes = ['Minimal', 'AllNps']
    POWER_REPEATS = 10
    full_results = {}
    for mm in map_modes:
        map_mode = getattr(akida.MapMode, mm)
        print(f'Running full-model benchmark (MapMode={mm}, {POWER_REPEATS} repeat(s))...')
        full_results[mm] = full_model_benchmark(
            akida_model, device, samples, map_mode=map_mode, repeats=POWER_REPEATS)

        akida_model.map(device, mode=map_mode)
        num_nps, num_passes, num_sequences = get_mapping_stats(akida_model)
        full_results[mm]['num_nps'] = num_nps
        full_results[mm]['num_passes'] = num_passes
        print(f'  Mapping: {num_nps} NP(s), {num_passes} pass(es), {num_sequences} sequence(s)')
        if num_sequences > 1:
            print('WARNING: model not completely mapped to hardware')

The plot below shows one column per map mode: a power trace (if a power meter
was connected) and the hardware mapping layout.

In [ ]:
if device is not None:
    plot_full_model_results(full_results, akida_model, device,
                            model_name=akida_model_path)

### Per-Layer Benchmark

Full-model timing tells us the total cost but not where time is spent.
`per_layer_benchmark` from
[brainchip_utils/hardware_utils.py](../../../brainchip_utils/hardware_utils.py)
reconstructs latency layer by layer by running cumulative sub-models and
differencing the results.

Because Akida processes events (non-zero activations), a layer's cost is
proportional to its *input* sparsity: a layer receiving 90% sparse inputs has
far fewer events to process than one receiving 10% sparse inputs. The per-layer
timing and the sparsity values computed above are therefore naturally correlated -
low-sparsity layers are typically the latency bottlenecks.

In [ ]:
from brainchip_utils.hardware_utils import per_layer_benchmark
from brainchip_utils.plot_utils import plot_per_layer_results

if device is not None:
    # Map without hw_only so akida_model.sequences is populated for the plot
    akida_model.map(device, mode=akida.MapMode.Minimal)

    print(f'Running per-layer benchmark ({len(samples)} samples)...')
    per_layer_results = per_layer_benchmark(akida_model, device, samples)

The plot stacks three panels: per-layer latency, input sparsity per layer, and
the hardware mapping. The inverse relationship between sparsity and latency is
the direct signature of the event-driven compute model: dense activations
generate more events, and more events mean more work for the hardware.

In [ ]:
if device is not None:
    plot_per_layer_results(per_layer_results, akida_model, sparsity_dict,
                           model_name=akida_model_path)